# 🚚 Food Delivery Clustering & Neural Network Prediction

**Student:** Hari Ganesh  
**Course:** Machine Learning  
**Dataset:** Food_Delivery_Time_Prediction.csv (200 records)

---

## 📌 Project Overview

This notebook applies **unsupervised learning** (K-Means & Hierarchical Clustering) to discover natural delivery groups, then builds a **Neural Network** to predict whether a delivery will be Fast or Delayed.

### Objectives
1. Preprocess and engineer features from raw delivery data
2. Find optimal delivery clusters using the Elbow Method
3. Visualise clusters using 2D scatter plots, location maps, and a 3D Plotly chart
4. Build and tune a Neural Network classifier
5. Compare model configurations and select the best one

---

## 📁 Pipeline

| Phase | Description |
|-------|-------------|
| **Phase 1** | Data Preprocessing & Feature Engineering |
| **Phase 2** | K-Means Clustering + Elbow Method |
| **Phase 3** | Hierarchical Clustering + Dendrogram |
| **Phase 4** | 3D Cluster Visualisation (Plotly) |
| **Phase 5** | Neural Network — Build & Train |
| **Phase 6** | Hyperparameter Tuning |
| **Phase 7** | Final Evaluation & Summary |


## Phase 1 — Data Preprocessing & Feature Engineering

### Step 1.1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

print("✅ All libraries imported successfully!")


### Step 1.2 — Load Dataset

In [ ]:
df = pd.read_csv('Food_Delivery_Time_Prediction.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()


### Step 1.3 — Data Overview & Quality Check

In [ ]:
print("=" * 50)
print("  DATASET INFORMATION")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("  MISSING VALUES")
print("=" * 50)
print(df.isnull().sum())

print("\n" + "=" * 50)
print("  BASIC STATISTICS")
print("=" * 50)
df.describe()


### Step 1.4 — Encode Categorical Columns

In [ ]:
Le = LabelEncoder()

df['Weather_Conditions'] = Le.fit_transform(df['Weather_Conditions'])
df['Traffic_Conditions'] = Le.fit_transform(df['Traffic_Conditions'])
df['Vehicle_Type']       = Le.fit_transform(df['Vehicle_Type'])
df['Order_Priority']     = Le.fit_transform(df['Order_Priority'])

print("✅ Label Encoding Applied:")
print("  Weather_Conditions → 0=Clear, 1=Fog, 2=Rain, 3=Stormy")
print("  Traffic_Conditions → 0=Low, 1=Medium, 2=High")
print("  Vehicle_Type       → 0=Bike, 1=Car, 2=Scooter")
print("  Order_Priority     → 0=High, 1=Low, 2=Medium")
df.head()


### Step 1.5 — Feature Engineering

#### 1.5a — Haversine Distance (Real GPS Distance in km)

In [ ]:
from haversine import haversine

# Extract latitude & longitude from string coordinates
df[['Customer_Latitude', 'Customer_Longitude']] = (
    df['Customer_Location']
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.split(', ', expand=True)
    .astype(float)
)

df[['Restaurant_Latitude', 'Restaurant_Longitude']] = (
    df['Restaurant_Location']
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.split(', ', expand=True)
    .astype(float)
)

# Calculate real-world distance using Haversine formula
df['Distance_km'] = df.apply(
    lambda row: haversine(
        (row['Customer_Latitude'],  row['Customer_Longitude']),
        (row['Restaurant_Latitude'], row['Restaurant_Longitude'])
    ), axis=1
)

print("✅ Haversine Distance calculated!")
print(f"   Min Distance : {df['Distance_km'].min():.2f} km")
print(f"   Max Distance : {df['Distance_km'].max():.2f} km")
print(f"   Mean Distance: {df['Distance_km'].mean():.2f} km")
df[['Customer_Latitude', 'Customer_Longitude', 'Restaurant_Latitude', 'Restaurant_Longitude', 'Distance_km']].head()


#### 1.5b — Order Time & Rush Hour Features

In [ ]:
# One-hot encode Order_Time
df['Order_Time_Morning']   = (df['Order_Time'] == 'Morning').astype(int)
df['Order_Time_Afternoon'] = (df['Order_Time'] == 'Afternoon').astype(int)
df['Order_Time_Evening']   = (df['Order_Time'] == 'Evening').astype(int)
df['Order_Time_Night']     = (df['Order_Time'] == 'Night').astype(int)

# Rush Hour = Morning, Evening, Night
df['Rush_Hour']     = ((df['Order_Time_Morning'] == 1) |
                       (df['Order_Time_Evening'] == 1) |
                       (df['Order_Time_Night']   == 1)).astype(int)
df['Non_Rush_Hour'] = (df['Rush_Hour'] == 0).astype(int)

print("✅ Rush Hour Feature Created!")
print(f"   Rush Hour orders    : {df['Rush_Hour'].sum()}")
print(f"   Non-Rush Hour orders: {df['Non_Rush_Hour'].sum()}")


### Step 1.6 — Scale Features for Clustering

In [ ]:
scaler = StandardScaler()

cluster_features = ['Delivery_Time', 'Distance_km', 'Traffic_Conditions', 'Weather_Conditions']
scaled_features  = scaler.fit_transform(df[cluster_features])

print("✅ StandardScaler applied to clustering features:")
for f in cluster_features:
    print(f"   {f}")
print(f"\nScaled array shape: {scaled_features.shape}")


## Phase 2 — K-Means Clustering

**What is K-Means?**  
K-Means groups data points into K clusters based on similarity.  
It works by minimising the distance between each point and its cluster centre (centroid).

**Steps:**
1. Use the **Elbow Method** to find the best K
2. Use **Silhouette Score** to validate cluster quality
3. Fit the final model and assign cluster labels


### Step 2.1 — Elbow Method (Find Optimal K)

In [ ]:
inertia        = []
silhouette_scores = []
k_range        = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled_features)
    inertia.append(km.inertia_)
    sil = silhouette_score(scaled_features, km.labels_)
    silhouette_scores.append(sil)

# Plot Elbow + Silhouette side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(k_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', label='Optimal K=4')
axes[0].set_title('Elbow Method — Optimal Number of Clusters', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='red', linestyle='--', label='K=4')
axes[1].set_title('Silhouette Score — Cluster Quality', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('K-Means: Choosing Optimal K', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n✅ Optimal K selected: 4")
print(f"   Silhouette Score at K=4: {silhouette_scores[2]:.4f}")


### Step 2.2 — Fit Final K-Means Model (K=4)

In [ ]:
kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans_model.fit_predict(scaled_features)

final_sil = silhouette_score(scaled_features, df['cluster'])

print("✅ K-Means Model Fitted!")
print(f"   Number of clusters : 4")
print(f"   Silhouette Score   : {final_sil:.4f}")
print(f"\n   Cluster Distribution:")
print(df['cluster'].value_counts().sort_index().to_string())


### Step 2.3 — Cluster Visualisations

#### 2.3a — Distance vs Delivery Time

In [ ]:
colors = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12']
labels = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

plt.figure(figsize=(10, 6))
for i in range(4):
    mask = df['cluster'] == i
    plt.scatter(df.loc[mask, 'Distance_km'],
                df.loc[mask, 'Delivery_Time'],
                c=colors[i], label=labels[i], alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

plt.title('K-Means Clusters: Distance vs Delivery Time', fontsize=14, fontweight='bold')
plt.xlabel('Distance (km)', fontsize=12)
plt.ylabel('Delivery Time', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### 2.3b — Traffic Conditions vs Delivery Time

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(4):
    mask = df['cluster'] == i
    plt.scatter(df.loc[mask, 'Traffic_Conditions'],
                df.loc[mask, 'Delivery_Time'],
                c=colors[i], label=labels[i], alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

plt.title('K-Means Clusters: Traffic Conditions vs Delivery Time', fontsize=14, fontweight='bold')
plt.xlabel('Traffic Conditions (0=Low, 1=Medium, 2=High)', fontsize=12)
plt.ylabel('Delivery Time', fontsize=12)
plt.legend(fontsize=11)
plt.xticks([0, 1, 2], ['Low', 'Medium', 'High'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### 2.3c — Weather Conditions vs Delivery Time

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(4):
    mask = df['cluster'] == i
    plt.scatter(df.loc[mask, 'Weather_Conditions'],
                df.loc[mask, 'Delivery_Time'],
                c=colors[i], label=labels[i], alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

plt.title('K-Means Clusters: Weather Conditions vs Delivery Time', fontsize=14, fontweight='bold')
plt.xlabel('Weather Conditions (0=Clear, 1=Fog, 2=Rain, 3=Stormy)', fontsize=12)
plt.ylabel('Delivery Time', fontsize=12)
plt.legend(fontsize=11)
plt.xticks([0, 1, 2, 3], ['Clear', 'Fog', 'Rain', 'Stormy'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### 2.3d — Customer & Restaurant Location Map by Cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i in range(4):
    mask = df['cluster'] == i
    axes[0].scatter(df.loc[mask, 'Customer_Longitude'],
                    df.loc[mask, 'Customer_Latitude'],
                    c=colors[i], label=labels[i], alpha=0.6, s=40)
    axes[1].scatter(df.loc[mask, 'Restaurant_Longitude'],
                    df.loc[mask, 'Restaurant_Latitude'],
                    c=colors[i], label=labels[i], alpha=0.6, s=40, marker='^')

axes[0].set_title('Customer Locations by Cluster', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Restaurant Locations by Cluster', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.suptitle('Geographic Distribution of Clusters', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 2.4 — Cluster Profile Analysis

Understanding what each cluster represents:


In [ ]:
cluster_profile = df.groupby('cluster')[
    ['Delivery_Time', 'Distance_km', 'Traffic_Conditions',
     'Weather_Conditions', 'Delivery_Person_Experience', 'Rush_Hour']
].mean().round(3)

cluster_profile.index = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']
cluster_profile.columns = ['Avg Delivery Time', 'Avg Distance (km)',
                            'Avg Traffic', 'Avg Weather',
                            'Avg Experience (yrs)', 'Rush Hour %']

print("=" * 70)
print("  CLUSTER PROFILE — MEAN VALUES PER CLUSTER")
print("=" * 70)
print(cluster_profile.to_string())
print()
print("Interpretation:")
print("  Cluster with HIGH delivery time + HIGH traffic → Delayed orders")
print("  Cluster with LOW delivery time + LOW distance  → Fast orders")


## Phase 3 — Hierarchical Clustering

**What is Hierarchical Clustering?**  
Instead of specifying K upfront, Hierarchical Clustering builds a tree (dendrogram) that shows how data points merge into clusters step by step.

**Ward Linkage Method:** Minimises the total within-cluster variance at each merge step — produces compact, well-separated clusters.


### Step 3.1 — Dendrogram

In [ ]:
linkage_matrix = linkage(scaled_features, method='ward')

plt.figure(figsize=(14, 7))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True,
    color_threshold=10
)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index (or Cluster Size)', fontsize=12)
plt.ylabel('Euclidean Distance', fontsize=12)
plt.axhline(y=10, color='red', linestyle='--', linewidth=1.5, label='Cut at 4 clusters')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("✅ Dendrogram plotted!")
print("   The red dashed line shows where we cut to get 4 clusters.")
print("   Branches below the line = final cluster groups.")


### Step 3.2 — Fit Agglomerative Clustering (4 Clusters)

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=4, linkage='ward')
df['agg_cluster'] = agg_model.fit_predict(scaled_features)

agg_sil = silhouette_score(scaled_features, df['agg_cluster'])

print("✅ Agglomerative Clustering Fitted!")
print(f"   Silhouette Score: {agg_sil:.4f}")
print(f"\n   Cluster Distribution:")
print(df['agg_cluster'].value_counts().sort_index().to_string())


### Step 3.3 — Compare K-Means vs Hierarchical Clustering

In [ ]:
kmeans_sil = silhouette_score(scaled_features, df['cluster'])
agg_sil    = silhouette_score(scaled_features, df['agg_cluster'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i in range(4):
    mask = df['cluster'] == i
    axes[0].scatter(df.loc[mask, 'Distance_km'],
                    df.loc[mask, 'Delivery_Time'],
                    c=colors[i], label=f'Cluster {i}', alpha=0.7, s=50)
axes[0].set_title(f'K-Means Clustering\nSilhouette Score: {kmeans_sil:.4f}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Distance (km)'); axes[0].set_ylabel('Delivery Time')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for i in range(4):
    mask = df['agg_cluster'] == i
    axes[1].scatter(df.loc[mask, 'Distance_km'],
                    df.loc[mask, 'Delivery_Time'],
                    c=colors[i], label=f'Cluster {i}', alpha=0.7, s=50)
axes[1].set_title(f'Hierarchical Clustering\nSilhouette Score: {agg_sil:.4f}',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Distance (km)'); axes[1].set_ylabel('Delivery Time')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('K-Means vs Hierarchical Clustering Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n{'Model':<30} {'Silhouette Score':>18}")
print("-" * 50)
print(f"{'K-Means (k=4)':<30} {kmeans_sil:>18.4f}")
print(f"{'Hierarchical (Ward, k=4)':<30} {agg_sil:>18.4f}")
better = 'K-Means' if kmeans_sil > agg_sil else 'Hierarchical'
print(f"\n✅ Better clustering method: {better}")


## Phase 4 — 3D Interactive Cluster Visualisation (Plotly)

A 3D scatter plot showing **Delivery Time × Distance × Traffic Conditions**, colour-coded by K-Means cluster.


In [ ]:
import plotly.graph_objs as go
import plotly.offline as py

cluster_colors_plotly = {0: 'red', 1: 'green', 2: 'blue', 3: 'orange'}

traces = []
for i in range(4):
    mask = df['cluster'] == i
    trace = go.Scatter3d(
        x=df.loc[mask, 'Delivery_Time'],
        y=df.loc[mask, 'Distance_km'],
        z=df.loc[mask, 'Traffic_Conditions'],
        mode='markers',
        name=f'Cluster {i}',
        marker=dict(
            color=cluster_colors_plotly[i],
            size=6,
            opacity=0.8,
            line=dict(color='white', width=0.5)
        )
    )
    traces.append(trace)

layout = go.Layout(
    title=dict(text='3D Cluster View: Delivery Time × Distance × Traffic', font=dict(size=16)),
    scene=dict(
        xaxis=dict(title='Delivery Time'),
        yaxis=dict(title='Distance (km)'),
        zaxis=dict(title='Traffic Conditions')
    ),
    legend=dict(x=0, y=1)
)

fig = go.Figure(data=traces, layout=layout)
py.iplot(fig)
print("✅ 3D Scatter Plot rendered above — rotate and zoom to explore clusters!")


## Phase 5 — Neural Network for Delivery Classification

**Goal:** Classify whether a delivery will be **Fast (1)** or **Delayed (0)** based on delivery features.

**Target definition:**  
- If `Delivery_Time > median` → **Delayed (1)**  
- If `Delivery_Time ≤ median` → **Fast (0)**

**Architecture:**
```
Input Layer  (15 features)
     ↓
Hidden Layer 1  (20 neurons, ReLU)
     ↓
Hidden Layer 2  (10 neurons, ReLU)
     ↓
Output Layer    (1 neuron, Sigmoid)
```


### Step 5.1 — Prepare Features & Target

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# Feature columns
feature_columns = [
    'Distance_km', 'Traffic_Conditions', 'Weather_Conditions',
    'Delivery_Person_Experience', 'Restaurant_Rating', 'Customer_Rating',
    'Order_Cost', 'Tip_Amount', 'Order_Time_Morning', 'Order_Time_Afternoon',
    'Order_Time_Evening', 'Order_Time_Night', 'Rush_Hour', 'Non_Rush_Hour',
    'Order_Priority'
]

X = df[feature_columns]

# Target: 1 = Delayed (above median), 0 = Fast
median_delivery_time = df['Delivery_Time'].median()
y = (df['Delivery_Time'] > median_delivery_time).astype(int)

print(f"✅ Target created using median threshold: {median_delivery_time:.4f}")
print(f"   Fast deliveries    (0): {(y == 0).sum()}")
print(f"   Delayed deliveries (1): {(y == 1).sum()}")
print(f"\n   Feature matrix shape: {X.shape}")


### Step 5.2 — Train/Test Split & Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

nn_scaler = StandardScaler()
X_train_scaled = nn_scaler.fit_transform(X_train)
X_test_scaled  = nn_scaler.transform(X_test)

print(f"✅ Train/Test Split Complete (80/20 stratified)")
print(f"   Training samples : {X_train_scaled.shape[0]}")
print(f"   Testing  samples : {X_test_scaled.shape[0]}")
print(f"   Features         : {X_train_scaled.shape[1]}")


### Step 5.3 — Build & Train Neural Network

In [ ]:
# Build model
nn_model = Sequential([
    Dense(20, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(10, activation='relu'),
    Dense(1,  activation='sigmoid')
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Neural Network Architecture:")
nn_model.summary()


In [ ]:
# Train
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)


### Step 5.4 — Training History Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='#2ECC71', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#E74C3C', linewidth=2)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', color='#3498DB', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#F39C12', linewidth=2)
axes[1].set_title('Model Loss Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Neural Network Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 5.5 — Evaluate Base Neural Network

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import numpy as np

predictions = nn_model.predict(X_test_scaled)
y_pred      = (predictions > 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("=" * 50)
print("  BASE NEURAL NETWORK — EVALUATION RESULTS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fast (0)', 'Delayed (1)']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fast', 'Predicted Delayed'],
            yticklabels=['Actual Fast', 'Actual Delayed'],
            linewidths=1, linecolor='white')
plt.title('Confusion Matrix — Base Neural Network', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Phase 6 — Hyperparameter Tuning

Testing different combinations of:
- **Layers:** 1 or 2 hidden layers
- **Neurons:** 16 or 32 per layer
- **Learning Rate:** 0.001 or 0.01

Total configurations tested: **8**


In [ ]:
from tensorflow.keras.optimizers import Adam

def build_model(layers, neurons, activation, lr, input_dim):
    model = Sequential()
    model.add(Dense(neurons, activation=activation, input_dim=input_dim))
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

layers_list    = [1, 2]
neurons_list   = [16, 32]
activations    = ['relu']
learning_rates = [0.001, 0.01]

best_acc   = 0
best_model = None
best_config = {}
results    = []

print(f"{'Layers':<8} {'Neurons':<10} {'LR':<8} {'Test Acc':>10}")
print("-" * 40)

for layers in layers_list:
    for neurons in neurons_list:
        for act in activations:
            for lr in learning_rates:
                m = build_model(layers, neurons, act, lr, input_dim=X_train_scaled.shape[1])
                m.fit(X_train_scaled, y_train, epochs=30, verbose=0, batch_size=16)
                _, acc = m.evaluate(X_test_scaled, y_test, verbose=0)
                results.append({'Layers': layers, 'Neurons': neurons, 'LR': lr, 'Accuracy': acc})
                print(f"{layers:<8} {neurons:<10} {lr:<8} {acc:>10.4f}")
                if acc > best_acc:
                    best_acc    = acc
                    best_model  = m
                    best_config = {'layers': layers, 'neurons': neurons, 'lr': lr}

print(f"\n✅ Best Configuration Found:")
print(f"   Layers  : {best_config['layers']}")
print(f"   Neurons : {best_config['neurons']}")
print(f"   LR      : {best_config['lr']}")
print(f"   Accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)")


### Step 6.1 — Hyperparameter Results Chart

In [ ]:
results_df = pd.DataFrame(results)
results_df['Config'] = (results_df['Layers'].astype(str) + 'L-' +
                        results_df['Neurons'].astype(str) + 'N-LR' +
                        results_df['LR'].astype(str))

plt.figure(figsize=(12, 5))
bar_colors = ['#2ECC71' if a == results_df['Accuracy'].max() else '#3498DB'
              for a in results_df['Accuracy']]
bars = plt.bar(results_df['Config'], results_df['Accuracy'], color=bar_colors, edgecolor='white', linewidth=1.5)
plt.axhline(y=results_df['Accuracy'].mean(), color='red', linestyle='--', linewidth=1.5, label='Average Accuracy')
plt.title('Hyperparameter Tuning — Test Accuracy per Configuration', fontsize=13, fontweight='bold')
plt.xlabel('Model Configuration (Layers-Neurons-LR)')
plt.ylabel('Test Accuracy')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.ylim(0, 1)
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


### Step 6.2 — Final Best Model Evaluation

In [ ]:
best_preds = best_model.predict(X_test_scaled)
best_y_pred = (best_preds > 0.5).astype(int)

b_acc  = accuracy_score(y_test, best_y_pred)
b_prec = precision_score(y_test, best_y_pred)
b_rec  = recall_score(y_test, best_y_pred)
b_f1   = f1_score(y_test, best_y_pred)

print("=" * 55)
print("  BEST NEURAL NETWORK — FINAL EVALUATION")
print("=" * 55)
print(f"  Configuration : {best_config['layers']} layers, "
      f"{best_config['neurons']} neurons, LR={best_config['lr']}")
print(f"  Accuracy      : {b_acc:.4f}  ({b_acc*100:.1f}%)")
print(f"  Precision     : {b_prec:.4f}")
print(f"  Recall        : {b_rec:.4f}")
print(f"  F1 Score      : {b_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, best_y_pred, target_names=['Fast (0)', 'Delayed (1)']))

# Confusion matrix
cm_best = confusion_matrix(y_test, best_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Predicted Fast', 'Predicted Delayed'],
            yticklabels=['Actual Fast', 'Actual Delayed'],
            linewidths=1, linecolor='white')
plt.title('Confusion Matrix — Best Neural Network', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Phase 7 — Final Summary & Comparison

In [ ]:
print("=" * 65)
print("  COMPLETE PROJECT SUMMARY")
print("=" * 65)

print("\n📦 DATASET")
print(f"   Records  : 200 rows × 15 columns")
print(f"   No missing values")

print("\n🔧 FEATURES ENGINEERED")
print("   Distance_km (Haversine), Rush_Hour, Non_Rush_Hour,")
print("   Order_Time dummies (Morning/Afternoon/Evening/Night)")

print("\n📊 CLUSTERING RESULTS")
print(f"   K-Means Silhouette Score        : {silhouette_score(scaled_features, df['cluster']):.4f}")
print(f"   Hierarchical Silhouette Score   : {silhouette_score(scaled_features, df['agg_cluster']):.4f}")
print(f"   Optimal clusters (both methods) : 4")

print("\n🧠 NEURAL NETWORK RESULTS")
print(f"   Base model accuracy             : {acc:.4f}  ({acc*100:.1f}%)")
print(f"   Best tuned model accuracy       : {b_acc:.4f}  ({b_acc*100:.1f}%)")
print(f"   Best config: {best_config['layers']} layers, "
      f"{best_config['neurons']} neurons, LR={best_config['lr']}")

print("\n💡 KEY INSIGHTS")
print("   1. K-Means clearly separates fast vs delayed delivery patterns")
print("   2. Rush hour and high traffic are strong signals for delayed clusters")
print("   3. Haversine distance is a more meaningful feature than raw Distance column")
print("   4. Neural Network performance is limited by small dataset (200 rows)")
print("   5. More training data would significantly improve model accuracy")

print("\n🔮 FUTURE IMPROVEMENTS")
print("   - Collect 1000+ records for better neural network training")
print("   - Try DBSCAN for non-circular cluster detection")
print("   - Add Dropout layers to reduce overfitting")
print("   - Use GridSearchCV with KerasClassifier for thorough tuning")
print("   - Build real-time prediction API using Flask or FastAPI")
print("=" * 65)
